# Predict Experiment

Train LSTM forecast artifacts, build shared data, and write prediction diagnostics for the explicit run.


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import os
import sys

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / 'configs' / 'cfg.py').exists())
os.chdir(_repo_root)
sys.path.insert(0, str(_repo_root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from configs.cfg import Cfg
from data.share_data import build_share_data
from predictors.lstm_eval import write_test_outputs_from_share_data
from predictors.lstm_training import train_forecasters
from scripts.plot import plot_predict_results
from utils.run_artifacts import create_run_dir, write_config_json

cfg = Cfg()
run_dir = create_run_dir(cfg)
forecast = train_forecasters(cfg, run_dir, overwrite=True)
cfg = replace(cfg, forecast=replace(cfg.forecast, lstm_artifact_dir=str(forecast['artifact_dir'])))
write_config_json(cfg, run_dir)
share_data_dir = build_share_data(cfg, run_dir, forecast['artifact_dir'], overwrite=True)
test_outputs = write_test_outputs_from_share_data(cfg, share_data_dir, run_dir)
figures = plot_predict_results(run_dir)

config_snapshot = pd.DataFrame([{
    'share_data_train_test': cfg.data.share_data_train_test,
    'train_start_date': cfg.data.train_start_date,
    'train_end_date': cfg.data.train_end_date,
    'eval_start_date': cfg.data.eval_start_date,
    'eval_end_date': cfg.data.eval_end_date,
    'history_window': cfg.forecast.history_window,
    'future_horizon': cfg.obs.sequence_length,
    'device': cfg.runtime.device,
    'artifact_dir': str(forecast['artifact_dir']),
}])
train_summary = pd.read_csv(run_dir / 'forecast' / 'tables' / 'forecast_train_summary.csv')
artifact_table = train_summary.groupby(['signal', 'component'], dropna=False, as_index=False).agg(artifact_count=('model_path', 'nunique'))
manifest = json.loads((share_data_dir / 'manifest.json').read_text(encoding='utf-8'))
manifest_summary = pd.DataFrame([{
    'schema_version': manifest['schema_version'],
    'pv_shared': manifest['pv_shared'],
    'train_episodes': manifest['splits']['train']['n_episodes'],
    'eval_episodes': manifest['splits']['eval']['n_episodes'],
    'share_data_dir': str(share_data_dir),
}])

display(Markdown(f'## Predict Results\\nRun dir: `{run_dir}`  \\nShare data: `{share_data_dir}`'))
display(config_snapshot)
display(artifact_table)
display(train_summary)
display(test_outputs['metrics'])
display(manifest_summary)
for figure in figures.values():
    display(figure)
    plt.close(figure)
